In [ ]:
pip install --upgrade transformers

In [2]:
import math
import random
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# 1) Perplexity

## Intuition & Motivation
Perplexity (PPL) is the most fundamental *intrinsic* metric for language models. It answers: **"How surprised is the model by this text?"**

Given a sequence of tokens  x = (x_1, x_2, …, x_N), a language model assigns a probability P(x_t | x_{<t}) to each token conditioned on its prefix. Perplexity exponentiates the average negative log-likelihood:

$$
PPL(x) = exp( −(1/N) Σ_{t=1}^{N} log P(x_t | x_{<t}) )
$$

**Why log-likelihood?**

Log-likelihood is additive across tokens, numerically stable, and directly optimized during pretraining (cross-entropy loss). Perplexity is simply its exponentiated, per-token average — it converts nats back into an intuitive "effective vocabulary size" the model is choosing from at each step.

**Lower PPL ⟹ better model** (on that data distribution). A perfect model that always assigns probability 1 to the true next token has PPL = 1. A uniform-random model over a vocab of size V has PPL = V.

### Sample Input → Output

    Input:  "The cat sat on the mat."
    Output: {
        "log_likelihood": -14.23,     # sum of per-token log probs
        "avg_neg_log_likelihood": 2.37, # = -log_likelihood / num_tokens
        "perplexity": 10.7             # = exp(avg_neg_log_likelihood)
    }

### Limitations
- PPL only measures how well the model predicts *this specific text*; it does not measure factuality, coherence, or instruction-following.
- Tokenizer choice affects the token count N, making cross-model PPL comparisons tricky.

In [4]:
# Small hand-crafted corpus: mix of fluent and disfluent sentences so we can
# observe that perplexity is higher for the disfluent / nonsensical ones.
eval_corpus = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning models require large amounts of data to generalize.",
    "Colorless green ideas sleep furiously.",                # Chomsky (grammatical but nonsensical)
    "The the the the the banana elephant.",                  # clearly broken
    "In recent years, large language models have achieved remarkable results.",
]

In [6]:
model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Loaded '{model_name}' on {device}")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded 'Qwen/Qwen3.5-0.8B' on cuda


In [7]:
# Per token log likelihood
def compute_token_log_probs(
    text: str,
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    device: torch.device,
) -> Tuple[List[float], List[str]]:
    """Compute the log-probability of every token in `text` under `model`."""

    # Tokenize the input text into token ids
    encoding = tokenizer(text, return_tensors="pt").to(device)
    # (1, seq_len)
    input_ids = encoding["input_ids"]

    with torch.no_grad():
        # Forward pass: get raw logits for every position
        # (1, seq_len) → (1, seq_len, vocab_size)
        outputs = model(**encoding)
        logits = outputs.logits

    # Convert logits to log-probabilities via log-softmax over vocabulary axis
    # (1, seq_len, vocab_size) → (1, seq_len, vocab_size)
    # -∞ to 0
    log_softmax = F.log_softmax(logits, dim=-1)

    # Predicted log-probs for positions 0..seq_len-2
    # (1, seq_len-1, vocab_size)
    pred_log_probs = log_softmax[:, :-1, :]

    # Target token ids at positions 1..seq_len-1
    # (1, seq_len-1)
    target_ids = input_ids[:, 1:]

    # Gather the log-prob assigned to each actual next token
    # (1, seq_len-1, vocab_size) → gather → (seq_len-1,)
    gathered = pred_log_probs.gather(
        dim=-1, index=target_ids.unsqueeze(-1)
    ).squeeze(-1).squeeze(0)

    log_probs = gathered.cpu().tolist()

    # Decode each target token for inspection
    tokens = [tokenizer.decode(tid) for tid in target_ids.squeeze(0).tolist()]

    return log_probs, tokens

In [9]:
# Perplexity Computation
def compute_perplexity(
    text: str,
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    device: torch.device,
) -> Dict[str, float]:
    """Compute perplexity of `text` under `model`."""
    log_probs, tokens = compute_token_log_probs(text, model, tokenizer, device)
    num_tokens = len(log_probs)

    total_log_likelihood = sum(log_probs)

    # the lower the better
    avg_nll = -total_log_likelihood / num_tokens
    perplexity = math.exp(min(avg_nll, 100.0))

    return {
        "text": text,
        "num_tokens": num_tokens,
        "log_likelihood": round(total_log_likelihood, 4),
        "avg_neg_log_likelihood": round(avg_nll, 4),
        "perplexity": round(perplexity, 4),
    }

In [10]:
print("\n" + "=" * 72)
print("PERPLEXITY EVALUATION")
print("=" * 72)
for sentence in eval_corpus:
    result = compute_perplexity(sentence, model, tokenizer, device)
    print(f"\n  Text : {result['text']}")
    print(f"  Tokens scored : {result['num_tokens']}")
    print(f"  Log-Likelihood: {result['log_likelihood']}")
    print(f"  Avg NLL       : {result['avg_neg_log_likelihood']}")
    print(f"  Perplexity    : {result['perplexity']}")


PERPLEXITY EVALUATION

  Text : The quick brown fox jumps over the lazy dog.
  Tokens scored : 9
  Log-Likelihood: -25.8984
  Avg NLL       : 2.8776
  Perplexity    : 17.7716

  Text : Machine learning models require large amounts of data to generalize.
  Tokens scored : 10
  Log-Likelihood: -24.321
  Avg NLL       : 2.4321
  Perplexity    : 11.3828

  Text : Colorless green ideas sleep furiously.
  Tokens scored : 7
  Log-Likelihood: -47.2422
  Avg NLL       : 6.7489
  Perplexity    : 853.1061

  Text : The the the the the banana elephant.
  Tokens scored : 7
  Log-Likelihood: -42.9219
  Avg NLL       : 6.1317
  Perplexity    : 460.2162

  Text : In recent years, large language models have achieved remarkable results.
  Tokens scored : 11
  Log-Likelihood: -30.1643
  Avg NLL       : 2.7422
  Perplexity    : 15.5212


# 2) N-gram Overlap Metrics: BLEU & ROUGE

## Intuition & Motivation
BLEU and ROUGE evaluate surface-level n-gram overlap between a hypothesis and a reference.

## 2.1 Bleu Score

In [11]:
def simple_tokenize(text: str) -> List[str]:
    """Lowercase and split on non-alphanumeric characters."""
    return re.findall(r"\w+", text.lower())

def extract_ngrams(tokens: List[str], n: int) -> Counter:
    """Return a Counter of all n-grams (as tuples) in `tokens`."""
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))

In [12]:
def compute_bleu(
    references: List[str],
    hypothesis: str,
    max_n: int = 4,
    weights: Optional[List[float]] = None,
) -> Dict[str, float]:
    """Compute corpus-level BLEU score (single hypothesis, multiple references)."""

    # With default BLEU-4, weights are [0.25, 0.25, 0.25, 0.25]
    if weights is None:
        weights = [1.0 / max_n] * max_n

    hyp_tokens = simple_tokenize(hypothesis)
    ref_token_lists = [simple_tokenize(ref) for ref in references]
    hyp_len = len(hyp_tokens)

    # minimum reference length
    ref_lens = [len(rt) for rt in ref_token_lists]
    closest_ref_len = min(ref_lens, key=lambda r: (abs(r - hyp_len), r))

    if hyp_len == 0:
        return {"bleu": 0.0, "brevity_penalty": 0.0, "precisions": [0.0] * max_n}

    # penalize short outputs, as a common and short word can have high precision
    if hyp_len < closest_ref_len:
        bp = math.exp(1.0 - closest_ref_len / hyp_len)
    else:
        bp = 1.0

    precisions = []
    for n in range(1, max_n + 1):
        hyp_ngrams = extract_ngrams(hyp_tokens, n)
        if len(hyp_ngrams) == 0:
            precisions.append(0.0)
            continue

        clipped_count = 0
        total_count = 0
        for ngram, count in hyp_ngrams.items():
            # Get the max freq of the particular ngam
            max_ref_count = max(extract_ngrams(rt, n).get(ngram, 0) for rt in ref_token_lists)
            clipped_count += min(count, max_ref_count)
            total_count += count
        precisions.append(clipped_count / total_count)

    log_avg_precision = 0.0
    for w, p in zip(weights, precisions):
        # return 0 if any of the ngram is 0
        if p == 0:
            return {"bleu": 0.0, "brevity_penalty": round(bp, 4), "precisions": [round(p, 4) for p in precisions]}
        log_avg_precision += w * math.log(p)

    bleu = bp * math.exp(log_avg_precision)
    return {"bleu": round(bleu, 4), "brevity_penalty": round(bp, 4), "precisions": [round(p, 4) for p in precisions]}

## 2.2 ROUGE

In [13]:
def compute_rouge_n(
    reference: str, hypothesis: str, n: int = 1
) -> Dict[str, float]:
    """Compute ROUGE-N: n-gram recall, precision, and F1 between reference
    and hypothesis.

    ROUGE-N recall = |overlapping n-grams| / |reference n-grams|
    ROUGE-N precision = |overlapping n-grams| / |hypothesis n-grams|
    F1 = harmonic mean of precision and recall

    Unlike BLEU, counts are NOT clipped
    Intersection of multisets (element-wise min) to avoid inflating overlap
    """

    ref_tokens = simple_tokenize(reference)
    hyp_tokens = simple_tokenize(hypothesis)

    ref_ngrams = extract_ngrams(ref_tokens, n)
    hyp_ngrams = extract_ngrams(hyp_tokens, n)

    # Multiset intersection: for each n-gram, take the minimum count
    overlap = 0
    for ngram, count in ref_ngrams.items():
        overlap += min(count, hyp_ngrams.get(ngram, 0))

    ref_count = sum(ref_ngrams.values())
    hyp_count = sum(hyp_ngrams.values())

    recall = overlap / ref_count if ref_count > 0 else 0.0
    precision = overlap / hyp_count if hyp_count > 0 else 0.0

    # The harmonic mean is the most punishing of the three (Arithmetic, Geometric, Harmonic)
    # It stays low unless both values are high. A single weak component drags the whole score down.
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        f"rouge_{n}_precision": round(precision, 4),
        f"rouge_{n}_recall": round(recall, 4),
        f"rouge_{n}_f1": round(f1, 4),
    }


def _lcs_length(a: List[str], b: List[str]) -> int:
    """Compute the length of the Longest Common Subsequence using dynamic
    programming.

    Time complexity: O(|a| * |b|).  Space could be reduced to O(min(|a|,|b|))
    but we keep the full table for clarity.
    """

    m, n = len(a), len(b)

    # DP table: dp[i][j] = length of LCS of a[:i] and b[:j]
    # (m+1, n+1) integer matrix
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    return dp[m][n]


def compute_rouge_l(reference: str, hypothesis: str) -> Dict[str, float]:
    """Compute ROUGE-L based on Longest Common Subsequence (LCS).

    ROUGE-L captures sentence-level structural similarity without requiring
    a fixed n-gram length.

    Recall    = LCS(ref, hyp) / len(ref)
    Precision = LCS(ref, hyp) / len(hyp)
    F1        = (1 + β²) * P * R / (R + β² * P),  where β = P/R
                (simplified to standard F1 when β → ∞, i.e., recall-weighted)

    Following Lin (2004), we use β = 1.2 (slightly recall-biased, standard
    for summarization).
    """

    ref_tokens = simple_tokenize(reference)
    hyp_tokens = simple_tokenize(hypothesis)
    lcs_len = _lcs_length(ref_tokens, hyp_tokens)

    ref_len = len(ref_tokens)
    hyp_len = len(hyp_tokens)

    recall = lcs_len / ref_len if ref_len > 0 else 0.0
    precision = lcs_len / hyp_len if hyp_len > 0 else 0.0

    beta = 1.2
    beta_sq = beta ** 2

    f1 = (
        (1 + beta_sq) * precision * recall / (recall + beta_sq * precision)
        if (recall + beta_sq * precision) > 0
        else 0.0
    )

    return {
        "rouge_l_precision": round(precision, 4),
        "rouge_l_recall": round(recall, 4),
        "rouge_l_f1": round(f1, 4),
    }

In [14]:
bleu_rouge_examples = [
    {
        "references": ["The cat is sitting on the mat."],
        "hypothesis": "The cat is on the mat.",
        "label": "Minor omission",
    },
    {
        "references": ["The cat is sitting on the mat."],
        "hypothesis": "A feline rests upon the rug.",
        "label": "Synonym paraphrase (no n-gram overlap)",
    },
    {
        "references": ["The cat is sitting on the mat."],
        "hypothesis": "The the the the the the the.",
        "label": "Degenerate repetition",
    },
    {
        "references": ["The cat is sitting on the mat."],
        "hypothesis": "The cat is sitting on the mat.",
        "label": "Exact match",
    },
]

print("\n" + "=" * 60)
print("BLEU & ROUGE EVALUATION")
print("=" * 60)

for ex in bleu_rouge_examples:
    print(f"\n  Label     : {ex['label']}")
    print(f"  Reference : {ex['references'][0]}")
    print(f"  Hypothesis: {ex['hypothesis']}")

    bleu = compute_bleu(ex["references"], ex["hypothesis"])
    r1 = compute_rouge_n(ex["references"][0], ex["hypothesis"], n=1)
    r2 = compute_rouge_n(ex["references"][0], ex["hypothesis"], n=2)
    rl = compute_rouge_l(ex["references"][0], ex["hypothesis"])

    print(f"  BLEU-4    : {bleu['bleu']}  (precisions={bleu['precisions']}, BP={bleu['brevity_penalty']})")
    print(f"  ROUGE-1 F1: {r1['rouge_1_f1']}")
    print(f"  ROUGE-2 F1: {r2['rouge_2_f1']}")
    print(f"  ROUGE-L F1: {rl['rouge_l_f1']}")


BLEU & ROUGE EVALUATION

  Label     : Minor omission
  Reference : The cat is sitting on the mat.
  Hypothesis: The cat is on the mat.
  BLEU-4    : 0.0  (precisions=[1.0, 0.8, 0.5, 0.0], BP=0.8465)
  ROUGE-1 F1: 0.9231
  ROUGE-2 F1: 0.7273
  ROUGE-L F1: 0.9104

  Label     : Synonym paraphrase (no n-gram overlap)
  Reference : The cat is sitting on the mat.
  Hypothesis: A feline rests upon the rug.
  BLEU-4    : 0.0  (precisions=[0.1667, 0.0, 0.0, 0.0], BP=0.8465)
  ROUGE-1 F1: 0.1538
  ROUGE-2 F1: 0.0
  ROUGE-L F1: 0.1517

  Label     : Degenerate repetition
  Reference : The cat is sitting on the mat.
  Hypothesis: The the the the the the the.
  BLEU-4    : 0.0  (precisions=[0.2857, 0.0, 0.0, 0.0], BP=1.0)
  ROUGE-1 F1: 0.2857
  ROUGE-2 F1: 0.0
  ROUGE-L F1: 0.2857

  Label     : Exact match
  Reference : The cat is sitting on the mat.
  Hypothesis: The cat is sitting on the mat.
  BLEU-4    : 1.0  (precisions=[1.0, 1.0, 1.0, 1.0], BP=1.0)
  ROUGE-1 F1: 1.0
  ROUGE-2 F1: 1.0
  RO

# 3) BERTScore

N-gram metrics fail catastrophically on paraphrases:
  - Reference:  "The cat is sitting on the mat."
  - Hypothesis: "A feline rests upon the rug."
  - BLEU ≈ 0, ROUGE-1 ≈ 0

BERTScore fixes this by comparing texts **in embedding space**:
1. Encode both reference and hypothesis with a pretrained encoder (e.g. BERT).
2. Compute a token-level cosine similarity matrix.
3. For each reference token, find its maximum-similarity hypothesis token (recall); for each hypothesis token, find its best reference token (precision).
4. Average these maxima to get R, P, F1.

### Key Design Decisions
- We use contextual embeddings (not static word2vec) because polysemy matters: "bank" in "river bank" ≠ "bank" in "savings bank".
- We apply IDF weighting (optional) to down-weight common tokens like "the", "is".  We implement the unweighted version for clarity.

Reference: Zhang et al., ICLR 2020.

In [15]:
encoder_name = "microsoft/deberta-base-mnli"
bert_tokenizer = AutoTokenizer.from_pretrained(encoder_name)
bert_model = AutoModel.from_pretrained(encoder_name)
bert_model.eval()
bert_model.to(device)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/557M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-base-mnli
Key                 | Status     |  | 
--------------------+------------+--+-
classifier.bias     | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 
config              | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DebertaModel(
  (embeddings): DebertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=0)
    (LayerNorm): DebertaLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): DebertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x DebertaLayer(
        (attention): DebertaAttention(
          (self): DisentangledSelfAttention(
            (in_proj): Linear(in_features=768, out_features=2304, bias=False)
            (pos_dropout): Dropout(p=0.1, inplace=False)
            (pos_proj): Linear(in_features=768, out_features=768, bias=False)
            (pos_q_proj): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): DebertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): DebertaLayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): DebertaIntermediate(
 

In [16]:
def get_token_embeddings(
    text: str,
    model: AutoModel,
    tokenizer: AutoTokenizer,
    device: torch.device,
    layer: int = -1,
) -> torch.Tensor:
    """Extract contextual token embeddings from a specified encoder layer.

    Parameters
    ----------
    layer : int
        Which hidden layer to use (-1 = last layer).  Zhang et al. [4]
        found that intermediate layers (e.g., layer 9 of BERT-base)
        sometimes work better; we default to the last layer for simplicity.

    Returns
    -------
    embeddings : Tensor of shape (num_tokens, embed_dim)
        One embedding per token, *excluding* [CLS] and [SEP].
    """

    encoding = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=512
    ).to(device)

    with torch.no_grad():
        # (1, seq_len) → hidden_states tuple, each (1, seq_len, embed_dim)
        outputs = model(**encoding, output_hidden_states=True)

    # Select the desired layer's hidden state
    # (1, seq_len, embed_dim)
    hidden = outputs.hidden_states[layer]

    # Remove batch dimension and strip [CLS] (first) and [SEP] (last) tokens
    # (seq_len, embed_dim) → (num_tokens, embed_dim)
    embeddings = hidden.squeeze(0)[1:-1]

    return embeddings

In [17]:
def compute_bertscore(
    reference: str,
    hypothesis: str,
    model: AutoModel,
    tokenizer: AutoTokenizer,
    device: torch.device,
) -> Dict[str, float]:
    """Compute BERTScore (precision, recall, F1) between a reference and
    hypothesis.

    Algorithm:
      1. Get contextual embeddings for reference and hypothesis.
      2. Build cosine similarity matrix S where S[i,j] = cos(ref_i, hyp_j).
      3. Recall:    (1/|ref|) * Σ_i max_j S[i,j]
         Precision: (1/|hyp|) * Σ_j max_i S[i,j]
      4. F1 = 2 * P * R / (P + R).

    This greedy matching has O(|ref| * |hyp|) complexity.
    """

    # Get contextual embeddings for both texts
    # (num_ref_tokens, embed_dim)
    ref_emb = get_token_embeddings(reference, model, tokenizer, device)

    # (num_hyp_tokens, embed_dim)
    hyp_emb = get_token_embeddings(hypothesis, model, tokenizer, device)

    # L2-normalize embeddings so dot product = cosine similarity
    # (num_ref_tokens, embed_dim) → (num_ref_tokens, embed_dim)
    ref_emb = F.normalize(ref_emb, p=2, dim=-1)

    # (num_hyp_tokens, embed_dim) → (num_hyp_tokens, embed_dim)
    hyp_emb = F.normalize(hyp_emb, p=2, dim=-1)

    # (num_ref_tokens, embed_dim) @ (embed_dim, num_hyp_tokens)
    # Output: (num_ref_tokens, num_hyp_tokens)
    sim_matrix = torch.mm(ref_emb, hyp_emb.t())

    # Recall: for each reference token, find its best-matching hypothesis token
    # (num_ref_tokens, num_hyp_tokens) → max over dim=1 → (num_ref_tokens,)
    recall_max_sim = sim_matrix.max(dim=1).values
    recall = recall_max_sim.mean().item()

    # Precision: for each hypothesis token, find its best-matching reference token
    # (num_ref_tokens, num_hyp_tokens) → max over dim=0 → (num_hyp_tokens,)
    precision_max_sim = sim_matrix.max(dim=0).values
    precision = precision_max_sim.mean().item()

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "bertscore_precision": round(precision, 4),
        "bertscore_recall": round(recall, 4),
        "bertscore_f1": round(f1, 4),
    }

In [18]:
bertscore_examples = [
    {
        "reference": "The cat is sitting on the mat.",
        "hypothesis": "A feline rests upon the rug.",
        "label": "Semantic paraphrase (low n-gram overlap)",
    },
    {
        "reference": "The cat is sitting on the mat.",
        "hypothesis": "Dogs are running in the park.",
        "label": "Unrelated content",
    },
    {
        "reference": "The cat is sitting on the mat.",
        "hypothesis": "The cat is sitting on the mat.",
        "label": "Exact match",
    },
]

print("\n" + "=" * 60)
print("BERTSCORE EVALUATION (compared with ROUGE-1 F1)")
print("=" * 60)
for ex in bertscore_examples:
    bs = compute_bertscore(
        ex["reference"], ex["hypothesis"], bert_model, bert_tokenizer, device
    )
    r1 = compute_rouge_n(ex["reference"], ex["hypothesis"], n=1)
    print(f"\n  Label        : {ex['label']}")
    print(f"  Reference    : {ex['reference']}")
    print(f"  Hypothesis   : {ex['hypothesis']}")
    print(f"  BERTScore F1 : {bs['bertscore_f1']}")
    print(f"  ROUGE-1 F1   : {r1['rouge_1_f1']}")
    print(f"  → BERTScore captures paraphrases that ROUGE misses."
          if bs["bertscore_f1"] > r1["rouge_1_f1"] + 0.1 else "")


BERTSCORE EVALUATION (compared with ROUGE-1 F1)


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con


  Label        : Semantic paraphrase (low n-gram overlap)
  Reference    : The cat is sitting on the mat.
  Hypothesis   : A feline rests upon the rug.
  BERTScore F1 : 0.8415
  ROUGE-1 F1   : 0.1538
  → BERTScore captures paraphrases that ROUGE misses.

  Label        : Unrelated content
  Reference    : The cat is sitting on the mat.
  Hypothesis   : Dogs are running in the park.
  BERTScore F1 : 0.7838
  ROUGE-1 F1   : 0.1538
  → BERTScore captures paraphrases that ROUGE misses.

  Label        : Exact match
  Reference    : The cat is sitting on the mat.
  Hypothesis   : The cat is sitting on the mat.
  BERTScore F1 : 1.0
  ROUGE-1 F1   : 1.0



# 3) Calibration & Expected Calibration Error (ECE)

A model is **well-calibrated** if, among all predictions where it says "I'm 80% confident the answer is A," roughly 80% actually are A.

For LLMs used in multiple-choice QA (MMLU, HellaSwag, etc.), we extract P(answer | question) from the model's logits and check: does the model "know what it knows"?

## 3.1 ECE

**Expected Calibration Error (ECE)** bins predictions by confidence and computes the weighted average gap between confidence and accuracy:

$$
ECE = Σ_{b=1}^{B} (|samples_in_b| / N) * |accuracy_b − confidence_b|
$$

**Sample:**

    Predictions: [(confidence=0.9, correct=True),
                  (confidence=0.8, correct=False),
                  (confidence=0.7, correct=True), ...]

    ECE = 0.12   (model is overconfident by ~12% on average)

### Why This Matters
- Overconfident models give dangerous false certainty in high-stakes applications (medicine, law).
- ECE is the standard calibration metric in the LLM literature.
- Post-hoc calibration (temperature scaling) can fix miscalibration without retraining — but you need ECE to diagnose the problem first.

Reference:
1. Guo et al., ICML 2017;
2. Hendrycks et al., ICLR 2021.

In [19]:
# Each question has 4 answer choices (A, B, C, D) and one correct answer.

synthetic_mcqa = [
    {"question": "What is the capital of France?",
     "choices": ["Berlin", "Paris", "Madrid", "Rome"], "answer": 1},
    {"question": "Which planet is closest to the Sun?",
     "choices": ["Venus", "Earth", "Mercury", "Mars"], "answer": 2},
    {"question": "What is 7 × 8?",
     "choices": ["54", "56", "58", "64"], "answer": 1},
    {"question": "Who wrote Romeo and Juliet?",
     "choices": ["Dickens", "Shakespeare", "Austen", "Hemingway"], "answer": 1},
    {"question": "What is the chemical symbol for water?",
     "choices": ["CO2", "NaCl", "H2O", "O2"], "answer": 2},
    {"question": "Which ocean is the largest?",
     "choices": ["Atlantic", "Indian", "Arctic", "Pacific"], "answer": 3},
    {"question": "What is the square root of 144?",
     "choices": ["10", "11", "12", "14"], "answer": 2},
    {"question": "In which year did World War II end?",
     "choices": ["1943", "1944", "1945", "1946"], "answer": 2},
]

In [20]:
def score_mcqa_choices(
    question: str,
    choices: List[str],
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    device: torch.device,
) -> np.ndarray:
    """Score each answer choice by its average log-probability given the question.

    For each choice, we form the sequence:
        "Question: {question}\nAnswer: {choice}"
    and compute the average log P(choice_token | prefix) over the choice
    tokens only. This is the standard "completion scoring" approach used
    in MMLU evaluations (see Hendrycks et al. [8], §4).

    Returns
    -------
    probs : np.ndarray of shape (num_choices,)
        Softmax-normalized probabilities over choices.
    """

    choice_log_probs = []

    for choice in choices:
        prompt = f"Question: {question}\nAnswer: {choice}"
        encoding = tokenizer(prompt, return_tensors="pt").to(device)
        # (1, seq_len)
        input_ids = encoding["input_ids"]

        # Determine how many tokens belong to the choice (answer portion)
        choice_encoding = tokenizer(choice, return_tensors="pt")
        num_choice_tokens = choice_encoding["input_ids"].size(1)

        with torch.no_grad():
            # (1, seq_len) → (1, seq_len, vocab_size)
            logits = model(**encoding).logits

        # (1, seq_len, vocab_size) → (1, seq_len, vocab_size)
        log_probs = F.log_softmax(logits, dim=-1)

        # Score only the choice tokens: the last `num_choice_tokens` tokens.
        total_len = input_ids.size(1)
        start = total_len - num_choice_tokens - 1
        end = total_len - 1

        # (num_choice_tokens, vocab_size)
        choice_pred_log_probs = log_probs[0, start:end, :]
        # (num_choice_tokens,)
        choice_target_ids = input_ids[0, start + 1 : end + 1]

        # Gather log-probs for the actual choice tokens
        # (num_choice_tokens, vocab_size) → gather → (num_choice_tokens,)
        token_lps = choice_pred_log_probs.gather(
            dim=-1, index=choice_target_ids.unsqueeze(-1)
        ).squeeze(-1)

        # Average log-probability per token (length-normalized)
        avg_lp = token_lps.mean().item()
        choice_log_probs.append(avg_lp)

    # Convert log-probs to probabilities via softmax for comparable scoring
    choice_log_probs = np.array(choice_log_probs)

    # Numerical stability: subtract max before exp
    shifted = choice_log_probs - choice_log_probs.max()
    probs = np.exp(shifted)
    probs = probs / probs.sum()

    return probs

In [21]:
def compute_ece(
    confidences: np.ndarray,
    correctness: np.ndarray,
    num_bins: int = 10,
) -> Dict[str, float]:
    """Compute Expected Calibration Error using equal-width binning.

    Parameters
    ----------
    confidences : np.ndarray of shape (num_samples,)
        Model's predicted probability for its top choice.
    correctness : np.ndarray of shape (num_samples,)
        Binary: 1 if the top choice was correct, 0 otherwise.
    num_bins : int
        Number of equal-width bins in [0, 1].

    Returns
    -------
    dict with ECE value and per-bin diagnostics.

    Implementation follows Guo et al. [5], Definition 1.
    """

    bin_boundaries = np.linspace(0.0, 1.0, num_bins + 1)
    bin_details = []
    ece = 0.0
    n_total = len(confidences)

    for b in range(num_bins):
        lo, hi = bin_boundaries[b], bin_boundaries[b + 1]

        # Select samples whose confidence falls in [lo, hi)  (last bin includes hi)
        if b == num_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        bin_size = mask.sum()

        if bin_size == 0:
            bin_details.append({
                "range": f"[{lo:.1f}, {hi:.1f})",
                "count": 0, "avg_conf": 0, "avg_acc": 0, "gap": 0,
            })
            continue

        avg_conf = confidences[mask].mean()
        avg_acc = correctness[mask].mean()
        gap = abs(avg_acc - avg_conf)

        # Weighted contribution to ECE
        ece += (bin_size / n_total) * gap

        bin_details.append({
            "range": f"[{lo:.1f}, {hi:.1f})",
            "count": int(bin_size),
            "avg_conf": round(float(avg_conf), 4),
            "avg_acc": round(float(avg_acc), 4),
            "gap": round(float(gap), 4),
        })

    return {"ece": round(float(ece), 4), "bins": bin_details}

In [22]:
print("\n" + "=" * 72)
print("CALIBRATION EVALUATION (MCQA)")
print("=" * 72)

all_confidences = []
all_correct = []

for item in synthetic_mcqa:
    probs = score_mcqa_choices(
        item["question"], item["choices"], model, tokenizer, device
    )
    predicted = int(np.argmax(probs))
    confidence = float(probs[predicted])
    correct = int(predicted == item["answer"])

    all_confidences.append(confidence)
    all_correct.append(correct)

    print(f"\n  Q: {item['question']}")
    print(f"  Probs: {[round(p, 3) for p in probs]}")
    print(f"  Predicted: {item['choices'][predicted]} (conf={confidence:.3f})")
    print(f"  Correct:   {item['choices'][item['answer']]}  {'✓' if correct else '✗'}")

all_confidences = np.array(all_confidences)
all_correct = np.array(all_correct)

ece_result = compute_ece(all_confidences, all_correct, num_bins=5)
accuracy = all_correct.mean()

print(f"\n  Overall Accuracy: {accuracy:.2%}")
print(f"  ECE (5 bins)    : {ece_result['ece']:.4f}")
print("\n  Per-bin breakdown:")
for b in ece_result["bins"]:
    if b["count"] > 0:
        print(f"    {b['range']:15s}  n={b['count']}  "
              f"avg_conf={b['avg_conf']:.3f}  avg_acc={b['avg_acc']:.3f}  "
              f"gap={b['gap']:.3f}")


CALIBRATION EVALUATION (MCQA)

  Q: What is the capital of France?
  Probs: [np.float64(0.023), np.float64(0.863), np.float64(0.002), np.float64(0.111)]
  Predicted: Paris (conf=0.863)
  Correct:   Paris  ✓

  Q: Which planet is closest to the Sun?
  Probs: [np.float64(0.227), np.float64(0.089), np.float64(0.564), np.float64(0.12)]
  Predicted: Mercury (conf=0.564)
  Correct:   Mercury  ✓

  Q: What is 7 × 8?
  Probs: [np.float64(0.027), np.float64(0.893), np.float64(0.024), np.float64(0.056)]
  Predicted: 56 (conf=0.893)
  Correct:   56  ✓

  Q: Who wrote Romeo and Juliet?
  Probs: [np.float64(0.017), np.float64(0.806), np.float64(0.013), np.float64(0.164)]
  Predicted: Shakespeare (conf=0.806)
  Correct:   Shakespeare  ✓

  Q: What is the chemical symbol for water?
  Probs: [np.float64(0.028), np.float64(0.005), np.float64(0.685), np.float64(0.282)]
  Predicted: H2O (conf=0.685)
  Correct:   H2O  ✓

  Q: Which ocean is the largest?
  Probs: [np.float64(0.342), np.float64(0.01), np.f

## 3.2 Temperature Scaling for Post-Hoc Calibration

### Temperature Scaling (Guo et al., 2017 [5])

Temperature scaling is the simplest post-hoc calibration method.  We learn a single scalar T > 0 and divide all logits by T before softmax:

$$
calibrated_probs = softmax(logits / T)
$$

- T > 1: softens the distribution → reduces overconfidence
- T < 1: sharpens → increases confidence
- T = 1: no change

We optimize T on a held-out validation set by minimizing NLL (equivalent to maximizing calibration).

In [23]:
def optimize_temperature(
    logits_list: List[np.ndarray],
    labels: List[int],
    lr: float = 0.01,
    num_iters: int = 200,
) -> float:
    """Find optimal temperature T via gradient descent on NLL.

    Parameters
    ----------
    logits_list : list of np.ndarray, each shape (num_choices,)
        Raw (unnormalized) logits for each sample.
    labels : list of int
        Correct answer index for each sample.

    Returns
    -------
    T : float
        Learned temperature parameter.
    """

    # Initialize T = 1.5 (slight softening as a prior; overconfidence is common)
    T = 1.5

    for iteration in range(num_iters):
        total_nll = 0.0
        grad_T = 0.0

        for logits, label in zip(logits_list, labels):
            # Scale logits by temperature
            # (num_choices,) → (num_choices,)
            scaled = logits / T

            # Numerically stable softmax
            shifted = scaled - scaled.max()
            exp_s = np.exp(shifted)
            probs = exp_s / exp_s.sum()

            # NLL for this sample
            total_nll -= np.log(probs[label] + 1e-12)

            # Gradient of NLL w.r.t. T:
            # d/dT [ -log softmax(z/T)_y ] = (1/T²) * (z_y - Σ_k p_k * z_k)
            # where z = logits, y = correct label
            # But we want to *minimize* NLL, so gradient is negative of above
            weighted_sum = (probs * logits).sum()
            grad_T += -(1.0 / (T * T)) * (logits[label] - weighted_sum)

        # Average gradients
        grad_T /= len(logits_list)

        # Gradient descent update
        T -= lr * grad_T

        # Clamp T to positive values
        T = max(T, 0.01)

    return T


# Collect raw logits for temperature scaling
raw_logits_list = []
labels_list = []

for item in synthetic_mcqa:
    choice_log_probs = []
    for choice in item["choices"]:
        prompt = f"Question: {item['question']}\nAnswer: {choice}"
        encoding = tokenizer(prompt, return_tensors="pt").to(device)
        choice_enc = tokenizer(choice, return_tensors="pt")
        n_ct = choice_enc["input_ids"].size(1)

        with torch.no_grad():
            logits = model(**encoding).logits
        lp = F.log_softmax(logits, dim=-1)
        total_len = encoding["input_ids"].size(1)
        s, e = total_len - n_ct - 1, total_len - 1
        c_lp = lp[0, s:e, :]
        c_ids = encoding["input_ids"][0, s + 1 : e + 1]
        avg = c_lp.gather(-1, c_ids.unsqueeze(-1)).squeeze(-1).mean().item()
        choice_log_probs.append(avg)

    raw_logits_list.append(np.array(choice_log_probs))
    labels_list.append(item["answer"])

# Optimize temperature (in production, use a held-out set; here we use the
# same data for demonstration purposes only)
optimal_T = optimize_temperature(raw_logits_list, labels_list)
print(f"\n  Optimal Temperature: {optimal_T:.4f}")

# Recompute calibrated predictions
calibrated_confs = []
calibrated_correct = []
for logits, label in zip(raw_logits_list, labels_list):
    scaled = logits / optimal_T
    shifted = scaled - scaled.max()
    probs = np.exp(shifted) / np.exp(shifted).sum()
    pred = int(np.argmax(probs))
    calibrated_confs.append(float(probs[pred]))
    calibrated_correct.append(int(pred == label))

calibrated_ece = compute_ece(
    np.array(calibrated_confs), np.array(calibrated_correct), num_bins=5
)
print(f"  ECE (before calibration): {ece_result['ece']:.4f}")
print(f"  ECE (after  calibration): {calibrated_ece['ece']:.4f}")


  Optimal Temperature: 1.9894
  ECE (before calibration): 0.2868
  ECE (after  calibration): 0.4610


# 4) LLM-as-a-Judge Framework

Human evaluation is the gold standard for open-ended generation quality, but it's expensive and slow.  Zheng et al. showed that strong LLMs (GPT-4 class) can serve as automated judges whose rankings correlate highly with human preferences (~80%+ agreement on MT-Bench).

**Key configurations:**
- **Pointwise scoring**: Judge rates a single response on a scale (1–5).
- **Pairwise comparison**: Judge picks a winner between two responses.
- **Reference-guided**: Judge sees a gold reference to guide its rating.

**Known biases**:
- **Position bias**: Judge prefers the first response in pairwise mode.
- **Verbosity bias**: Longer responses rated higher regardless of quality.
- **Self-enhancement bias**: A model rates its own outputs more favorably.

Mitigation: swap positions and average, enforce structured output, use rubrics.

Reference: Zheng et al., NeurIPS 2023.

In [24]:
judge_eval_data = [
    {
        "prompt": "Explain quantum computing in simple terms.",
        "response_a": (
            "Quantum computing uses qubits that can be both 0 and 1 at the "
            "same time thanks to superposition. This lets quantum computers "
            "try many solutions simultaneously for certain problems."
        ),
        "response_b": (
            "Quantum computing is a type of computing. It is very fast. "
            "It uses quantum mechanics. It is the future of technology."
        ),
        "human_preference": "A",
    },
    {
        "prompt": "Write a haiku about machine learning.",
        "response_a": "Data flows like streams / Gradients descend the hills / Models learn to see",
        "response_b": "Machine learning is / A very important field / We should study it",
        "human_preference": "A",
    },
    {
        "prompt": "What are the ethical concerns with AI?",
        "response_a": (
            "Key ethical concerns include algorithmic bias that perpetuates "
            "discrimination, job displacement in automation-vulnerable sectors, "
            "privacy erosion through mass surveillance, and existential risks "
            "from misaligned superintelligent systems."
        ),
        "response_b": (
            "AI has some concerns. People worry about it. We should be careful "
            "with AI. It could be dangerous."
        ),
        "human_preference": "A",
    },
]

## 4.1 Pointwise Judge

In [34]:
def build_pointwise_judge_prompt(
    prompt: str,
    response: str,
    rubric: Optional[str] = None,
) -> str:
    """Construct a structured prompt for pointwise evaluation.

    The prompt engineering follows best practices from [6]:
    - Explicit rubric with score anchors
    - Chain-of-thought reasoning before the score
    - Structured output format for reliable parsing

    We provide a default rubric covering: relevance, accuracy, depth, and
    clarity — the four dimensions used in MT-Bench [6].
    """

    if rubric is None:
        rubric = (
            "Evaluate the response on four dimensions:\n"
            "1. Relevance: Does it address the prompt directly?\n"
            "2. Accuracy: Is the information factually correct?\n"
            "3. Depth: Does it provide sufficient detail and nuance?\n"
            "4. Clarity: Is it well-organized and easy to understand?\n\n"
            "Score anchors:\n"
            "  5 = Excellent on all dimensions\n"
            "  4 = Good with minor issues\n"
            "  3 = Acceptable but lacks depth or has some inaccuracies\n"
            "  2 = Poor: major issues with relevance or accuracy\n"
            "  1 = Very poor: off-topic, incoherent, or completely wrong"
        )

    judge_prompt = (
        f"You are an expert evaluator. Rate the following response.\n\n"
        f"[Rubric]\n{rubric}\n\n"
        f"[User Prompt]\n{prompt}\n\n"
        f"[Response]\n{response}\n\n"
        f"Provide your evaluation in this exact format:\n"
        f"REASONING: <your chain-of-thought analysis>\n"
        f"SCORE: <integer from 1 to 5>"
    )
    return judge_prompt

def parse_judge_output(text: str) -> Dict[str, any]:
    """Parse the judge's structured output to extract score and reasoning.

    Robust parsing with fallback: we search for the SCORE pattern anywhere
    in the output, since models may deviate from the exact format.
    """

    score = None
    reasoning = ""

    # Extract reasoning
    reasoning_match = re.search(
        r"REASONING:\s*(.*?)(?=SCORE:|$)", text, re.DOTALL
    )
    if reasoning_match:
        reasoning = reasoning_match.group(1).strip()

    # Extract score (integer 1-5)
    score_match = re.search(r"SCORE:\s*(\d)", text)
    if score_match:
        s = int(score_match.group(1))
        if 1 <= s <= 5:
            score = s

    return {"score": score, "reasoning": reasoning, "raw_output": text}

def llm_pointwise_judge(
    prompt: str,
    response: str,
    rubric: Optional[str] = None,
) -> Dict:
    """Uses the provided LLM to perform pointwise judgment."""
    judge_prompt = build_pointwise_judge_prompt(prompt, response, rubric)

    # Encode the prompt
    input_ids = tokenizer(judge_prompt, return_tensors="pt").input_ids.to(device)

    # Generate a response from the LLM
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=200,  # Limit generation length
            do_sample=False,     # Use greedy decoding for reproducibility
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode the generated text, excluding the input prompt
    generated_text = tokenizer.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)

    # Parse the LLM's output
    parsed_output = parse_judge_output(generated_text)
    parsed_output["raw_output"] = generated_text # Store raw for debugging
    return parsed_output

## 4.2 Pairwise Judge

In [35]:
def build_pairwise_judge_prompt(
    prompt: str,
    response_a: str,
    response_b: str,
) -> str:
    """Construct a pairwise comparison prompt.

    Position bias mitigation: In practice, we would call this twice with
    A and B swapped and aggregate.  Here we build one direction; the
    debiasing logic is in `compute_pairwise_judgment_debiased`.
    """

    judge_prompt = (
        f"You are an expert evaluator. Compare two responses to a prompt "
        f"and decide which is better.\n\n"
        f"[User Prompt]\n{prompt}\n\n"
        f"[Response A]\n{response_a}\n\n"
        f"[Response B]\n{response_b}\n\n"
        f"Consider: relevance, accuracy, depth, and clarity.\n\n"
        f"Provide your evaluation in this exact format:\n"
        f"REASONING: <your analysis comparing both responses>\n"
        f"WINNER: <A or B or TIE>"
    )

    return judge_prompt


def parse_pairwise_output(text: str) -> Dict[str, any]:
    """Parse pairwise judge output to extract winner and reasoning."""

    reasoning = ""
    winner = None

    reasoning_match = re.search(
        r"REASONING:\s*(.*?)(?=WINNER:|$)", text, re.DOTALL
    )
    if reasoning_match:
        reasoning = reasoning_match.group(1).strip()

    winner_match = re.search(r"WINNER:\s*(A|B|TIE)", text, re.IGNORECASE)
    if winner_match:
        winner = winner_match.group(1).upper()

    return {"winner": winner, "reasoning": reasoning, "raw_output": text}

def llm_pairwise_judge(
    prompt: str,
    response_a: str,
    response_b: str,
) -> Dict:
    """Uses the provided LLM to perform pairwise judgment."""
    judge_prompt = build_pairwise_judge_prompt(prompt, response_a, response_b)

    # Encode the prompt
    input_ids = tokenizer(judge_prompt, return_tensors="pt").input_ids.to(device)

    # Generate a response from the LLM
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=200,  # Limit generation length
            do_sample=False,     # Use greedy decoding
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode the generated text, excluding the input prompt
    generated_text = tokenizer.decode(
        output_ids[0][input_ids.shape[1]:],
        skip_special_tokens=True
    )

    # Parse the LLM's output
    parsed_output = parse_pairwise_output(generated_text)
    parsed_output["raw_output"] = generated_text # Store raw for debugging
    return parsed_output

## 4.3 Position Bias Mitigation

In [37]:
def compute_pairwise_judgment_debiased(
    prompt: str,
    response_a: str,
    response_b: str,
    judge_fn,
) -> Dict:
    """Run pairwise judgment in both orderings and aggregate to mitigate
    position bias.

    Strategy (from Zheng et al. [6], §4):
    - Run judge(A, B) and judge(B, A)
    - If both agree on the same winner → use that
    - If they disagree → declare TIE (conservative)
    - This eliminates first-position bias at the cost of 2× judge calls.
    """

    # First ordering: A first, B second
    result_ab = judge_fn(prompt, response_a, response_b)
    winner_ab = result_ab["winner"]

    # Second ordering: B first, A second
    result_ba = judge_fn(prompt, response_b, response_a)
    # Map the winner back: if judge said "A" (which is now B), map to "B"
    winner_ba_raw = result_ba["winner"]
    if winner_ba_raw == "A":
        winner_ba = "B"
    elif winner_ba_raw == "B":
        winner_ba = "A"
    else:
        winner_ba = "TIE"

    # Aggregate: consistent ⟹ accept; inconsistent ⟹ TIE
    if winner_ab == winner_ba:
        final_winner = winner_ab
        consistent = True
    else:
        final_winner = "TIE"
        consistent = False

    return {
        "winner": final_winner,
        "winner_ordering_1": winner_ab,
        "winner_ordering_2": winner_ba,
        "consistent": consistent,
    }

## 4.4 Inter-Annotator Agreement (Cohen's Kappa)

In [38]:
def compute_cohens_kappa(
    annotations_1: List[str], annotations_2: List[str]
) -> float:
    """Compute Cohen's Kappa for inter-annotator agreement.

    Cohen's Kappa adjusts raw agreement for the probability of chance:
        κ = (p_observed − p_chance) / (1 − p_chance)

    Values:
        κ < 0    → worse than chance
        κ = 0    → chance agreement
        κ = 0.41–0.60 → moderate
        κ = 0.61–0.80 → substantial
        κ > 0.80 → almost perfect

    We use this to measure how well the LLM judge agrees with human labels.
    """

    assert len(annotations_1) == len(annotations_2)
    n = len(annotations_1)

    # Collect all unique labels
    labels = sorted(set(annotations_1) | set(annotations_2))

    # Build confusion matrix
    matrix = {}
    for l1 in labels:
        for l2 in labels:
            matrix[(l1, l2)] = 0

    for a1, a2 in zip(annotations_1, annotations_2):
        matrix[(a1, a2)] += 1

    # Observed agreement: proportion of exact matches
    p_observed = sum(matrix[(l, l)] for l in labels) / n

    # Expected (chance) agreement
    p_chance = 0.0
    for label in labels:
        # Proportion of times annotator 1 chose this label
        p1 = sum(matrix[(label, l2)] for l2 in labels) / n
        # Proportion of times annotator 2 chose this label
        p2 = sum(matrix[(l1, label)] for l1 in labels) / n
        p_chance += p1 * p2

    if p_chance == 1.0:
        return 1.0

    kappa = (p_observed - p_chance) / (1.0 - p_chance)
    return round(kappa, 4)

## 4.5 Run LLM-as-a-Judge Evaluation

In [39]:
print("\n" + "=" * 60)
print("LLM-AS-A-JUDGE EVALUATION")
print("=" * 60)

judge_winners = []
human_labels = []

for ex in judge_eval_data:
    # Pointwise scores for both responses
    score_a = llm_pointwise_judge(ex["prompt"], ex["response_a"])
    score_b = llm_pointwise_judge(ex["prompt"], ex["response_b"])

    # Debiased pairwise comparison
    pairwise = compute_pairwise_judgment_debiased(
        ex["prompt"], ex["response_a"], ex["response_b"],
        llm_pairwise_judge,
    )

    judge_winners.append(pairwise["winner"])
    human_labels.append(ex["human_preference"])

    print(f"\n  Prompt: {ex['prompt'][:60]}...")
    print(f"  Pointwise A: {score_a['score']}/5  |  Pointwise B: {score_b['score']}/5")
    print(f"  Pairwise Winner: {pairwise['winner']}  (consistent={pairwise['consistent']})")
    print(f"  Human Preference: {ex['human_preference']}")

# Compute agreement between judge and humans
kappa = compute_cohens_kappa(judge_winners, human_labels)
raw_agreement = sum(j == h for j, h in zip(judge_winners, human_labels)) / len(human_labels)
print(f"\n  Raw Agreement (judge vs human): {raw_agreement:.2%}")
print(f"  Cohen's Kappa:                  {kappa}")


LLM-AS-A-JUDGE EVALUATION

  Prompt: Explain quantum computing in simple terms....
  Pointwise A: None/5  |  Pointwise B: None/5
  Pairwise Winner: TIE  (consistent=False)
  Human Preference: A

  Prompt: Write a haiku about machine learning....
  Pointwise A: None/5  |  Pointwise B: None/5
  Pairwise Winner: TIE  (consistent=False)
  Human Preference: A

  Prompt: What are the ethical concerns with AI?...
  Pointwise A: None/5  |  Pointwise B: None/5
  Pairwise Winner: TIE  (consistent=False)
  Human Preference: A

  Raw Agreement (judge vs human): 0.00%
  Cohen's Kappa:                  0.0


# 5) Elo Rating for Model Comparison

When comparing K > 2 models, pairwise win rates don't compose transitively.
Elo ratings — originally designed for chess — provide a principled way to derive a single scalar strength estimate per model from pairwise matches.

The Chatbot Arena leaderboard (Zheng et al. ) uses Elo to rank LLMs from crowd-sourced pairwise human preferences. The system works as follows:

After each match between models A and B:
    E_A = 1 / (1 + 10^((R_B − R_A) / 400))    # expected win prob for A
    R_A ← R_A + K * (S_A − E_A)                 # update rating
where S_A = 1 (win), 0.5 (tie), 0 (loss), and K is the update step size.

### Sample Input → Output

    Match results: [(A beats B), (A beats C), (B beats C), (C beats A)]
    →  Elo Ratings: {A: 1520, B: 1500, C: 1480}

### Properties
- Converges to a stable ordering with enough matches.
- Sensitive to match order (mitigated by bootstrapping — see §6.3).
- K factor controls sensitivity: large K ⟹ volatile, small K ⟹ stable.

Reference: Elo, 1978 [7]; Zheng et al., NeurIPS 2023 [6].

## 5.1 Elo Rating System

In [40]:
class EloRatingSystem:
    """Implements the Elo rating system for pairwise model comparison.

    Attributes
    ----------
    ratings : dict[str, float]
        Current rating for each model. Initialized to `initial_rating`.
    k_factor : float
        Controls how much a single match can change a rating.
        Chatbot Arena uses K=4 for stability; we use K=32 for faster
        convergence on our small dataset.
    initial_rating : float
        Starting rating for all models (conventionally 1500).
    history : list[dict]
        Log of all matches for analysis.
    """

    def __init__(self, k_factor: float = 32, initial_rating: float = 1500.0):
        self.k_factor = k_factor
        self.initial_rating = initial_rating
        self.ratings: Dict[str, float] = {}
        self.history: List[Dict] = []

    def _ensure_registered(self, model: str):
        """Register a model with the initial rating if not yet seen."""
        if model not in self.ratings:
            self.ratings[model] = self.initial_rating

    def expected_score(self, rating_a: float, rating_b: float) -> float:
        """Compute the expected score (win probability) for player A.

        E_A = 1 / (1 + 10^((R_B − R_A) / 400))

        This is the logistic function with base 10 and scale 400 —
        a 400-point rating gap corresponds to a 10:1 win odds.
        """
        return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))

    def record_match(
        self, model_a: str, model_b: str, winner: str
    ):
        """Update ratings based on a single pairwise match.

        Parameters
        ----------
        winner : str
            "A", "B", or "TIE".
        """
        self._ensure_registered(model_a)
        self._ensure_registered(model_b)

        r_a = self.ratings[model_a]
        r_b = self.ratings[model_b]

        # Expected scores
        e_a = self.expected_score(r_a, r_b)
        e_b = 1.0 - e_a

        # Actual scores
        if winner == "A":
            s_a, s_b = 1.0, 0.0
        elif winner == "B":
            s_a, s_b = 0.0, 1.0
        else:
            s_a, s_b = 0.5, 0.5

        # Elo update rule
        self.ratings[model_a] = r_a + self.k_factor * (s_a - e_a)
        self.ratings[model_b] = r_b + self.k_factor * (s_b - e_b)

        self.history.append({
            "model_a": model_a, "model_b": model_b, "winner": winner,
            "rating_a_before": round(r_a, 2),
            "rating_b_before": round(r_b, 2),
            "rating_a_after": round(self.ratings[model_a], 2),
            "rating_b_after": round(self.ratings[model_b], 2),
        })

    def get_leaderboard(self) -> List[Tuple[str, float]]:
        """Return models sorted by rating (descending)."""
        return sorted(self.ratings.items(), key=lambda x: -x[1])

In [41]:
# Synthetic match data representing a hypothetical arena with 4 models of
# different quality tiers.  In production, these matches come from human
# annotators or LLM-judge evaluations.
synthetic_matches = [
    # Strong model (A) generally beats others
    ("model_strong", "model_medium", "A"),
    ("model_strong", "model_medium", "A"),
    ("model_strong", "model_weak", "A"),
    ("model_strong", "model_weak", "A"),
    ("model_strong", "model_tiny", "A"),
    # Medium model beats weak/tiny, occasionally ties strong
    ("model_medium", "model_weak", "A"),
    ("model_medium", "model_weak", "A"),
    ("model_medium", "model_tiny", "A"),
    ("model_medium", "model_tiny", "A"),
    ("model_medium", "model_strong", "TIE"),
    # Weak model beats tiny
    ("model_weak", "model_tiny", "A"),
    ("model_weak", "model_tiny", "A"),
    # Upsets (realistic noise)
    ("model_weak", "model_strong", "A"),
    ("model_tiny", "model_medium", "A"),
    # More matches for stability
    ("model_strong", "model_tiny", "A"),
    ("model_strong", "model_medium", "A"),
    ("model_medium", "model_weak", "A"),
    ("model_weak", "model_tiny", "A"),
    ("model_strong", "model_weak", "A"),
    ("model_medium", "model_tiny", "A"),
]

elo = EloRatingSystem(k_factor=32)
for model_a, model_b, winner in synthetic_matches:
    elo.record_match(model_a, model_b, winner)

print("\n" + "=" * 72)
print("ELO RATING LEADERBOARD")
print("=" * 72)
for rank, (model_id, rating) in enumerate(elo.get_leaderboard(), 1):
    print(f"  #{rank}  {model_id:20s}  Elo = {rating:.1f}")


ELO RATING LEADERBOARD
  #1  model_strong          Elo = 1586.8
  #2  model_medium          Elo = 1524.8
  #3  model_weak            Elo = 1479.6
  #4  model_tiny            Elo = 1408.9


## 5.2 Bootstrap Confidence Intervals for Elo

### Bootstrapped Elo Confidence Intervals

Elo ratings are sensitive to match ordering. To quantify uncertainty, Chatbot Arena uses bootstrapping:
1. Resample the set of matches with replacement.
2. Recompute Elo ratings from scratch on each bootstrap sample.
3. Report the 2.5th and 97.5th percentiles as a 95% confidence interval.

This is a non-parametric method that makes no distributional assumptions.

In [42]:
def bootstrap_elo(
    matches: List[Tuple[str, str, str]],
    k_factor: float = 32,
    num_bootstrap: int = 500,
) -> Dict[str, Dict[str, float]]:
    """Compute bootstrapped Elo ratings with confidence intervals.

    Parameters
    ----------
    matches : list of (model_a, model_b, winner) tuples
    num_bootstrap : int
        Number of bootstrap resamples. Chatbot Arena uses 1000+; we use
        500 for speed in this demo.

    Returns
    -------
    dict mapping model name → {mean, median, ci_lower, ci_upper}
    """

    all_ratings: Dict[str, List[float]] = {}

    for _ in range(num_bootstrap):
        # Resample matches with replacement
        resampled = [matches[i] for i in np.random.randint(0, len(matches), len(matches))]

        elo_system = EloRatingSystem(k_factor=k_factor)
        for model_a, model_b, winner in resampled:
            elo_system.record_match(model_a, model_b, winner)

        for model_id, rating in elo_system.ratings.items():
            if model_id not in all_ratings:
                all_ratings[model_id] = []
            all_ratings[model_id].append(rating)

    # Compute statistics
    results = {}
    for model_id, ratings in all_ratings.items():
        ratings = np.array(ratings)
        results[model_id] = {
            "mean": round(float(ratings.mean()), 1),
            "median": round(float(np.median(ratings)), 1),
            "ci_lower": round(float(np.percentile(ratings, 2.5)), 1),
            "ci_upper": round(float(np.percentile(ratings, 97.5)), 1),
        }

    return results

bootstrap_results = bootstrap_elo(synthetic_matches, k_factor=32, num_bootstrap=500)

print("\n  Bootstrapped Elo (500 resamples, 95% CI):")
# Sort by mean rating
for model_id, stats in sorted(bootstrap_results.items(), key=lambda x: -x[1]["mean"]):
    print(
        f"    {model_id:20s}  mean={stats['mean']:.1f}  "
        f"[{stats['ci_lower']:.1f}, {stats['ci_upper']:.1f}]"
    )


  Bootstrapped Elo (500 resamples, 95% CI):
    model_strong          mean=1586.9  [1528.6, 1640.5]
    model_medium          mean=1524.9  [1461.1, 1586.8]
    model_weak            mean=1477.7  [1411.0, 1546.4]
    model_tiny            mean=1410.5  [1357.6, 1470.4]


# Summary

### When to Use What

| Scenario                    | Recommended Metrics                    |
|-----------------------------|----------------------------------------|
| Comparing LM pretraining    | Perplexity                             |
| Machine translation          | BLEU + BERTScore                       |
| Summarization                | ROUGE + BERTScore                      |
| MCQA benchmarks (MMLU)       | Accuracy + ECE                         |
| Open-ended chat quality      | LLM-Judge + Elo (pairwise arena)       |
| Safety / factuality audits   | Human eval + LLM-Judge (reference-guided)|

### Further Reading
- Holtzman et al., "The Curious Case of Neural Text Degeneration," 2020 (on the gap between PPL and generation quality).
- Wang et al., "Is ChatGPT a Good NLG Evaluator?", 2023 (systematic study of LLM judges).
- Liang et al., "Holistic Evaluation of Language Models (HELM)," 2023 (multi-metric, multi-scenario benchmarking framework).

In [32]:
print("\n" + "=" * 72)
print("NOTEBOOK COMPLETE")
print("=" * 72)
print("All evaluation metrics implemented from scratch.")
print("See the Summary section for guidance on metric selection.")


NOTEBOOK COMPLETE
All evaluation metrics implemented from scratch.
See the Summary section for guidance on metric selection.
